## Import bibliotek

In [38]:
import requests

In [39]:
pip install BeautifulSoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
import requests
from bs4 import BeautifulSoup

Pobranie z serwisu Ceneo.pl opinii o wybranym produkcie


Potencjalne produkty:
- 7690465
- 187595793
- 10247
- 130492450
- 47293506

In [42]:
headers = {
    "Cookie": "sv3=1.0_766fc797-1e16-11f1-bb99-1d4020d53c9d",
    "user-agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36 OPR/128.0.0.0",
    "host": "www.ceneo.pl"
}

In [43]:
url = "https://www.ceneo.pl/47293506?fto=601413596&se=q9t0K4trRu9YunYeFywfj3fmh46Jrlju&gad_source=1&gad_campaignid=21301561911&gclid=CjwKCAjwyMnNBhBNEiwA-Kcgu1FR9CI6zppFsNlIhzWE7k4pe6OSnaKQoKa9FEUqBTpxp5wR-P3h-hoCYuMQAvD_BwE"
response = requests.get(url)
print(response.status_code)

200


In [44]:
page_dom = BeautifulSoup(response.text, "html.parser")

In [45]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


In [46]:
opinion = page_dom.select_one("div.js_product-review")
print(type(opinion))

<class 'bs4.element.Tag'>


In [47]:
opinion = opinions.pop(0)
print(type(opinion))
print(opinion)

<class 'bs4.element.Tag'>
<div class="user-post user-post__card js_product-review" data-entry-id="6572033">
<div class="user-post__header">
<div class="js_lazy user-post__avatar user-rank__avatar" data-bg="https://lh3.googleusercontent.com/a/ACg8ocJ1IHOBJlk6FG0_qf5bk7OjwA0wsi_tLhzonDVIUtkJ9jjfwkA=s96-c"></div>
</div>
<div class="user-post__body">
<div class="user-post__content">
<div>
<span class="user-post__author-name">
Paweł
</span>
<span class="user-post__score">
<span class="screen-reader-text">Ocena:</span>
<span class="score-container score-container--s js_score-container">
<span class="score-marker score-marker--s" style="width: 100.0%"></span>
</span>
<span class="user-post__score-count">5/5</span>
<span class="user-post__author-recomendation">
<em class="recommended">Polecam</em>
</span>
</span>
</div>
<div class="mb-4">
<span class="verified-purchase">
Zaufana Opinia potwierdzona zakupem
</span>
<span class="user-post__published">
<span class="user-post__published">
Wystawio

| składowa | nazwa | selektor |
|---|---|---|
| opinia | opinion | div.js_product-review:not(.user-post--highlight) |
| identyfikator | review_id | [data-entry-id] |
| autor | author | span.user-post__author-name |
| treść | content | div.user-post__text |
| ocena | rating | span.user-post__score-count |
| rekomendacja | recommendation | span.user-post__author-recomendation > em |
| lista zalet | pros | div.review-feature__item--positives ~ div.review-feature__item |
| lista wad | cons | div.review-feature__title--negatives ~ div.review-feature__item |
| dla ilu przydatna | useful | button.vote-yes > span |
| dla ilu nieprzydatna | useless | button.vote-no > span |
| data zamieszczenia | publish_date | span.user-post__published > time:nth-child(1)[datetime] |
| data zakupu | purchase_date | span.user-post__published > time:nth-child(2)[datetime] |

In [ ]:
all_opinions = []
for opinion in opinions:
    opinion_data = {}
    
    # identyfikator opinii
    opinion_data["id"] = opinion.get("data-entry-id")
    
    # Autor
    opinion_data["author"] = opinion.select_one(".user-post__author-name").get_text(strip=True)
    
    # Treść opinii
    opinion_data["content"] = opinion.select_one("div.user-post__text").get_text(strip=True)
    
    # Ocena
    opinion_data["score"] = opinion.select_one("span.user-post__score-count").get_text(strip=True)
    opinion_data["score"] = float(opinion_data["score"].split("/")[0].replace(",", "."))

    
    # Rekomendacja (Polecam/Nie polecam)

    try:
        opinion_data["recommendation"] = opinion.select_one("span.user-post__author-recomendation > em").get_text(strip=True)
    except AttributeError:
        opinion_data["recommendation"] = NotImplemented

    # Zalety
    opinion_data["pros"] = [p.get_text(strip=True) for p in opinion.select("div.review-feature__item--positive")]

    
    # Wady
    opinion_data["cons"] = [c.get_text(strip=True) for c in opinion.select("div.review-feature__item--negative")]

    
    # Dla ilu osób przydatna
    opinion_data["useful"] = int(opinion.select_one("button.vote-yes > span").get_text(strip=True))
                                
    # Dla ilu osób nieprzydatna
    opinion_data["unuseful"] = int(opinion.select_one("button.vote-no > span").get_text(strip=True))
    
    # Data zamieszczenia
    opinion_data["published_date"] = opinion.select_one("span.user-post__published > time:nth-child(1)").get("datetime")
    
    # Data zakupu / potwierdzony zakup
    try:
        opinion_data["purchase_date"] = opinion.select_one("span.user-post__published > time:nth-child(2)").get("datetime")
    except AttributeError:
        opinion_data["purchase_date"] = None
    
    all_opinions.append(opinion_data)

# Wyświetlenie zebranych opinii
for single_opinion in all_opinions:
    print(single_opinion)





{'id': '16601645', 'author': 'k...r', 'content': 'Myszka jest rewelacyjna. To jakby porównać odgłos klawiatury z czasów WIN95 z współczesną cichutką klawiaturą laptopową. I jedno i drugie będzie działało ale po co słuchać tego irytującego klikania. A pomnóż  to np. x10 osób w biurze:) \nBardzo udany zakup.', 'score': 5.0, 'recommendation': 'Polecam', 'pros': [], 'cons': [], 'useful': 0, 'unsueful': 0, 'published_date': '2022-10-06 11:51:00', 'pucharse_date': '2022-09-30 08:22:14'}
{'id': '16601645', 'author': 'k...r', 'content': 'Myszka jest rewelacyjna. To jakby porównać odgłos klawiatury z czasów WIN95 z współczesną cichutką klawiaturą laptopową. I jedno i drugie będzie działało ale po co słuchać tego irytującego klikania. A pomnóż  to np. x10 osób w biurze:) \nBardzo udany zakup.', 'score': 5.0, 'recommendation': 'Polecam', 'pros': [], 'cons': [], 'useful': 0, 'unsueful': 0, 'published_date': '2022-10-06 11:51:00', 'pucharse_date': '2022-09-30 08:22:14'}
{'id': '19734571', 'author':

In [49]:
page = 1
next = True
while next:
    url = f"https://www.ceneo.pl/47293506/opinie-{page}"
    response = requests.get(url, headers=headers)
    page_dom = BeautifulSoup(response.text, "html.parser")
    opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
    print(f"{url} => {response.status_code} => {len(opinions)}")
    next = True if page_dom.select_one("button.pagination__next") else False
    page += 1

https://www.ceneo.pl/47293506/opinie-1 => 200 => 10
https://www.ceneo.pl/47293506/opinie-2 => 200 => 0
